In [106]:
import os, numpy as np

os.chdir(r'C:\SML_Projects\SML_project_5')

In [107]:
import pandas as pd
from joblib import load
from src.preprocessing import Preprocessing
from src.feature_engineering import FeatureEngineering
from sklearn.metrics import r2_score, mean_absolute_error

# User data

In [108]:
offline_data = pd.DataFrame({
    'Id': [102],
    'Title': ['Paddington 2'],
    'Year': [2017],
    'Duration': [103],
    'Rated': ['PG'],
    'IMDb_Rating': [7.8],
    'Votes': [150000],
    'Director': ['Paul King'],
    'Stars': ['Hugh Grant, Ben Whishaw, Sally Hawkins'],
    'Description': [
        "Paddington, now happily settled with the Brown family, picks up a series of odd jobs to buy the perfect present for his Aunt Lucy's 100th birthday, only for the gift to be stolen."
    ],
    'Genre': [None]
})


df_original = offline_data.copy()
df = offline_data.copy()

# Fill missing values

In [109]:
preprocessing = Preprocessing(df_original, target="Genre")
df = preprocessing.fillMissingValues().getDataset()

In [110]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Id           1 non-null      object
 1   Year         1 non-null      object
 2   Duration     1 non-null      object
 3   IMDb_Rating  1 non-null      object
 4   Votes        1 non-null      object
 5   Title        1 non-null      object
 6   Rated        1 non-null      object
 7   Director     1 non-null      object
 8   Stars        1 non-null      object
 9   Description  1 non-null      object
 10  Genre        0 non-null      object
dtypes: object(11)
memory usage: 220.0+ bytes


# Feature creation

In [111]:
fc = FeatureEngineering(df)
df = fc.create_tfidf_features().getDataset()
print("after feature_creation:", df.columns.tolist())

after feature_creation: ['Id', 'Year', 'Duration', 'IMDb_Rating', 'Votes', 'Title', 'Rated', 'Director', 'Stars', 'Description', 'Genre', 'Description_length', 'Duration_min', 'Duration_category', 'text_combined', 'tfidf_0', 'tfidf_1', 'tfidf_2', 'tfidf_3', 'tfidf_4', 'tfidf_5', 'tfidf_6', 'tfidf_7', 'tfidf_8', 'tfidf_9', 'tfidf_10', 'tfidf_11', 'tfidf_12', 'tfidf_13', 'tfidf_14', 'tfidf_15', 'tfidf_16', 'tfidf_17', 'tfidf_18', 'tfidf_19', 'tfidf_20', 'tfidf_21', 'tfidf_22', 'tfidf_23', 'tfidf_24', 'tfidf_25', 'tfidf_26', 'tfidf_27']


In [112]:
tfidf_vectorizer = load(r'model/best/tfidf_vectorizer.joblib')
X_tfidf = tfidf_vectorizer.transform(df['text_combined'])
tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=[f"tfidf_{i}" for i in range(X_tfidf.shape[1])])
tfidf_df.index = df.index
df = pd.concat([df, tfidf_df], axis=1)

# Preprocessing

In [113]:
df = preprocessing.encode().scale().getDataset()
df = preprocessing.logTransformation(df)

print("after encode/scale/log:", df.columns.tolist())

after encode/scale/log: ['Id', 'Year', 'Duration', 'IMDb_Rating', 'Votes', 'Title', 'Rated', 'Director', 'Stars', 'Description', 'Genre']


In [114]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Id           1 non-null      float64
 1   Year         1 non-null      float64
 2   Duration     1 non-null      float64
 3   IMDb_Rating  1 non-null      float64
 4   Votes        1 non-null      float64
 5   Title        1 non-null      float64
 6   Rated        1 non-null      float64
 7   Director     1 non-null      float64
 8   Stars        1 non-null      float64
 9   Description  1 non-null      float64
 10  Genre        1 non-null      int8   
dtypes: float64(10), int8(1)
memory usage: 213.0 bytes


In [115]:
model = load(r'model/best/HistGradientBoostingClassifier.joblib')

In [116]:
x_new = df.drop(columns=['Genre'], errors='ignore')

for col in model.feature_names_in_:
    if col not in x_new.columns:
        x_new[col] = 0
x_new = x_new[model.feature_names_in_]

C:\Users\user\AppData\Local\Temp\ipykernel_23564\2657568339.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  x_new[col] = 0
C:\Users\user\AppData\Local\Temp\ipykernel_23564\2657568339.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  x_new[col] = 0
C:\Users\user\AppData\Local\Temp\ipykernel_23564\2657568339.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instea

In [117]:
y_pred = model.predict(x_new)
df_original['Predicted_Genre'] = y_pred

In [118]:
genre_mapping = {0: 'Action', 1: 'Family', 2: 'Sport', 3: 'Horror'}

df_original['Predicted_Genre'] = pd.Series(y_pred).map(genre_mapping)

In [119]:
for idx, row in df_original.iterrows():
    print(f"Id: {row['Id']} | Title: {row['Title']} | Predicted Genre: {row['Predicted_Genre']}")

Id: 102 | Title: Paddington 2 | Predicted Genre: Horror
